# Student Performance Dataset Cleaning

Clean and validate the messy student performance dataset.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

input_path = "Student_Performance_Messy_50.csv"
output_path = "Student_Performance_Cleaned_50.csv"

df = pd.read_csv(input_path)
print("Loaded shape:", df.shape)

Loaded shape: (51, 8)


## Standardize Text and Missing Values

In [2]:
str_cols = [
    col for col in df.columns
    if pd.api.types.is_string_dtype(df[col])
]
for col in str_cols:
    df[col] = df[col].astype("string").str.strip()

missing_tokens = {"", "na", "n/a", "null", "none", "unknown", "not available", "-"}
for col in str_cols:
    df[col] = df[col].mask(df[col].str.lower().isin(missing_tokens))

print("Missing values after standardization:")
print(df.isna().sum())

Missing values after standardization:
StudentID      0
StudentName    0
Gender         0
Department     0
Python         3
SQL            4
Statistics     3
ExamDate       0
dtype: int64


## Normalize Categorical Columns

In [3]:
df["Gender"] = (
    df["Gender"]
    .str.upper()
    .map({"F": "Female", "FEMALE": "Female", "M": "Male", "MALE": "Male"})
)

print(df["Gender"].value_counts(dropna=False))

Gender
Male      27
Female    24
Name: count, dtype: int64


In [4]:
df["Department"] = df["Department"].str.title()

print("Departments:", sorted(df["Department"].dropna().unique()))

Departments: ['Computer Science', 'Data Science', 'Electronics', 'Information Technology', 'Mechanical']


In [5]:
print("Departments:", sorted(df["Department"].dropna().unique()))

Departments: ['Computer Science', 'Data Science', 'Electronics', 'Information Technology', 'Mechanical']


## Clean Scores and Fill Missing Values

In [6]:
score_cols = ["Python", "SQL", "Statistics"]
df["Python"] = pd.to_numeric(df["Python"], errors="coerce")
df.loc[~df["Python"].between(0, 100), "Python"] = np.nan
df["Python"] = df["Python"].fillna(df["Python"].median())

print(df["Python"].describe())

count    51.000000
mean     62.205882
std      16.649978
min      36.000000
25%      53.000000
50%      58.500000
75%      76.000000
max      97.000000
Name: Python, dtype: float64


In [7]:
df["SQL"] = pd.to_numeric(df["SQL"], errors="coerce")
df.loc[~df["SQL"].between(0, 100), "SQL"] = np.nan
df["SQL"] = df["SQL"].fillna(df["SQL"].median())

print(df["SQL"].describe())

count    51.000000
mean     66.823529
std      18.845377
min      35.000000
25%      47.500000
50%      66.000000
75%      83.000000
max      98.000000
Name: SQL, dtype: float64


In [8]:
df["Statistics"] = pd.to_numeric(df["Statistics"], errors="coerce")
df.loc[~df["Statistics"].between(0, 100), "Statistics"] = np.nan
df["Statistics"] = df["Statistics"].fillna(df["Statistics"].median())

print(df["Statistics"].describe())

count    51.000000
mean     70.000000
std      16.243152
min      38.000000
25%      56.000000
50%      71.000000
75%      84.000000
max      98.000000
Name: Statistics, dtype: float64


## Normalize Exam Dates

In [9]:
df["ExamDate"] = pd.to_datetime(
    df["ExamDate"],
    format="mixed",
    errors="coerce"
)

print("Invalid or missing dates:", df["ExamDate"].isna().sum())
df["ExamDate"] = df["ExamDate"].dt.strftime("%Y-%m-%d")

Invalid or missing dates: 3


## Remove Duplicate Students and Validate

In [10]:
duplicate_count = df["StudentID"].duplicated().sum()
df = df.drop_duplicates(subset="StudentID", keep="first").reset_index(drop=True)

assert df["StudentID"].is_unique
assert df["Gender"].dropna().isin(["Female", "Male"]).all()
assert df[score_cols].apply(lambda col: col.between(0, 100).all()).all()
assert df["ExamDate"].isna().sum() <= 3

print("Duplicates removed:", duplicate_count)
print("Missing values:")
print(df.isna().sum())

Duplicates removed: 1
Missing values:
StudentID      0
StudentName    0
Gender         0
Department     0
Python         0
SQL            0
Statistics     0
ExamDate       2
dtype: int64


## Save the Cleaned Dataset

In [ ]:
df.to_csv(output_path, index=False)

cleaned_df = pd.read_csv(output_path)
print("Saved to:", output_path)
print("Saved shape:", cleaned_df.shape)

Saved to: c:\Users\Dharshan\Desktop\Data_Cleaning\Student performance\Student_Performance_Cleaned_50.csv
Saved shape: (50, 8)
